> **Archived research notebook.** Retained for reproducibility and comparisons; it may require the historical code/dependencies. Use the [current notebook catalog](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/README.md) for new runs.


# SmolLM2-135M → Memory Fusion r48 / r64

This notebook creates **two full SmolLM2 students** in which all 30 Transformer self-attention modules are progressively replaced by TinyCeNN **Memory Fusion** attention.

Memory Fusion combines:
- adaptive + MaxPool Cellular local/multiscale attention,
- Hedgehog-style positive global linear memory,
- GDN2-style editable delta memory,
- token-wise learned fusion.

The original SmolLM2 FFNs are kept unchanged so the experiment isolates the attention replacement.

Training mirrors `SmolLM2_AMCeNN_Top2_v2_Colab.ipynb`:
1. replace attention in small groups,
2. directly distill each new attention module on the same teacher hidden states,
3. replace all 30 attention modules,
4. run end-to-end CE + logit-KL + hidden-state distillation,
5. save/reload the custom model,
6. verify that no Transformer attention remains,
7. compare r48/r64 with original SmolLM2 on the same held-out WikiText-2 blocks,
8. run generation smoke tests.

The custom Memory Fusion core runs in float32 for numerical stability; SmolLM2 projections use bf16/fp16 on GPU.


In [1]:
import subprocess, sys, pathlib, importlib, json, os, math, gc, time
import torch

subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR),
    'huggingface_hub', 'datasets', 'pandas'
], check=True)
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
import tinycenn_lm
print('TinyCeNN import:', tinycenn_lm.__file__)


TinyCeNN import: /content/TinyCeNN-LM/src/tinycenn_lm/__init__.py


## Hugging Face login

Publishing is optional. If you want the trained r48/r64 checkpoints uploaded, store a **write token** in Colab Secrets as `HF_TOKEN`.


In [2]:
from huggingface_hub import HfApi, login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

HF_USER = None
api = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi(token=HF_TOKEN)
    HF_USER = api.whoami()['name']
    print('HF user:', HF_USER)
else:
    print('HF_TOKEN not set: training/evaluation will work; publishing will be skipped.')


HF user: vtava


## Settings


In [3]:
BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'
MEMORY_RANKS = [48, 64]
FEATURE_DIM = 32
CONTEXT_LENGTH = 128
GROUP_SIZE = 5

# Balanced defaults. Increase these after the first successful run if desired.
CALIBRATION_STEPS = 30
FINAL_MAX_TOKENS = 350_000
MAX_RUNTIME_MINUTES_PER_RANK = 60
SEED = 73

OUTPUT_ROOT = REPO_DIR / 'checkpoints' / 'smollm2-memory-fusion'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_REPOS = {
    rank: (f'{HF_USER}/SmolLM2-135M-MemoryFusion-r{rank}' if HF_USER else None)
    for rank in MEMORY_RANKS
}
print('base:', BASE_MODEL)
print('ranks:', MEMORY_RANKS)
print('outputs:', OUTPUT_ROOT)


base: HuggingFaceTB/SmolLM2-135M
ranks: [48, 64]
outputs: /content/TinyCeNN-LM/checkpoints/smollm2-memory-fusion


## Train r48 and r64

Each rank is trained as a separate full student. All 30 attention layers are progressively replaced. Pretrained Q/K/V projections remain frozen; the Memory Fusion cores and output projections are calibrated and globally distilled.


In [4]:
reports = {}
for rank in MEMORY_RANKS:
    output_dir = OUTPUT_ROOT / f'r{rank}'
    cmd = [
        sys.executable, str(REPO_DIR / 'scripts' / 'train_smollm2_memory_fusion.py'),
        '--base-model', BASE_MODEL,
        '--output-dir', str(output_dir),
        '--context-length', str(CONTEXT_LENGTH),
        '--feature-dim', str(FEATURE_DIM),
        '--memory-rank', str(rank),
        '--group-size', str(GROUP_SIZE),
        '--calibration-steps', str(CALIBRATION_STEPS),
        '--final-max-tokens', str(FINAL_MAX_TOKENS),
        '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES_PER_RANK),
        '--seed', str(SEED),
    ]
    print('\n' + '=' * 100)
    print('TRAINING MEMORY FUSION r' + str(rank))
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
    report_path = output_dir / 'smollm2_memory_fusion_training_report.json'
    reports[rank] = json.loads(report_path.read_text())

print('\nTraining complete.')



TRAINING MEMORY FUSION r48
/usr/bin/python3 /content/TinyCeNN-LM/scripts/train_smollm2_memory_fusion.py --base-model HuggingFaceTB/SmolLM2-135M --output-dir /content/TinyCeNN-LM/checkpoints/smollm2-memory-fusion/r48 --context-length 128 --feature-dim 32 --memory-rank 48 --group-size 5 --calibration-steps 30 --final-max-tokens 350000 --max-runtime-minutes 60 --seed 73

[TinyCeNN][START] train_smollm2_memory_fusion
[TinyCeNN][COMMAND] /usr/bin/python3 /content/TinyCeNN-LM/scripts/train_smollm2_memory_fusion.py --base-model HuggingFaceTB/SmolLM2-135M --output-dir /content/TinyCeNN-LM/checkpoints/smollm2-memory-fusion/r48 --context-length 128 --feature-dim 32 --memory-rank 48 --group-size 5 --calibration-steps 30 --final-max-tokens 350000 --max-runtime-minutes 60 --seed 73
[TinyCeNN][OUTPUT] /content/TinyCeNN-LM/checkpoints/smollm2-memory-fusion/r48
[TinyCeNN][LOCAL LOG] /content/.colab_live_backup/train_smollm2_memory_fusion-20260914T223756Z/train.log
[TinyCeNN][BACKUP REQUIRED] https://

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


[TinyCeNN][BACKUP] uploading successful final checkpoint...
----------------------------------------------------------------------------------------
[TinyCeNN][BACKUP COMPLETE] private Hugging Face backup committed for train_smollm2_memory_fusion-20260914T233843Z
[TinyCeNN][DONE] train_smollm2_memory_fusion completed in 1h 00m 23s


Training complete.


## Compare training/calibration reports


In [5]:
import pandas as pd
rows = []
for rank, report in reports.items():
    stages = report['calibration_stages']
    rows.append({
        'rank': rank,
        'calibration_tokens': report['calibration_tokens'],
        'global_tokens': report['final_seen_tokens'],
        'global_updates': report['final_updates'],
        'last_CE': report['last_training_ce'],
        'last_KL': report['last_distillation_kl'],
        'last_hidden': report['last_hidden_alignment'],
        'last_stage_NMSE': stages[-1]['last_nmse'],
        'last_stage_cosine_distance': stages[-1]['last_cosine_distance'],
        'elapsed_min': report['elapsed_minutes'],
        'peak_VRAM_GiB': report['peak_vram_gib'],
        'trainable_params': report['parameters']['trainable'],
    })
training_df = pd.DataFrame(rows).sort_values('rank')
display(training_df)


,rank,calibration_tokens,global_tokens,global_updates,last_CE,last_KL,last_hidden,last_stage_NMSE,last_stage_cosine_distance,elapsed_min,peak_VRAM_GiB,trainable_params
0,48,23040,47872,93,5.896353,4.375960,0.899062,0.725541,0.305854,60.069441,3.295241,18083970
1,64,23040,48896,95,6.132222,3.689799,0.733257,0.725944,0.305957,60.011832,3.753481,19760130


## Optional: publish both custom models to Hugging Face


In [6]:
if api is None:
    print('Skipping upload because HF_TOKEN is not configured.')
else:
    for rank in MEMORY_RANKS:
        repo_id = TARGET_REPOS[rank]
        output_dir = OUTPUT_ROOT / f'r{rank}'
        readme = f'''---
library_name: transformers
base_model: {BASE_MODEL}
tags:
- tinycenn
- memory-fusion
- recurrent-memory
- causal-language-model
---

# SmolLM2-135M Memory Fusion r{rank}

Research checkpoint from TinyCeNN-LM. All 30 original self-attention modules are replaced by the custom Memory Fusion layer (adaptive-MaxPool Cellular + Hedgehog-style global memory + GDN2-style editable memory). Original FFNs are retained.

Load with `tinycenn_lm.smollm2_memory_fusion.build_smollm2_memory_fusion`. The current reference implementation requires `use_cache=False`.
'''
        (output_dir / 'README.md').write_text(readme)
        api.create_repo(repo_id, repo_type='model', exist_ok=True)
        api.upload_folder(
            repo_id=repo_id,
            repo_type='model',
            folder_path=str(output_dir),
            commit_message=f'Publish SmolLM2 Memory Fusion r{rank}',
        )
        print('Published:', f'https://huggingface.co/{repo_id}')


Published: https://huggingface.co/vtava/SmolLM2-135M-MemoryFusion-r48
Published: https://huggingface.co/vtava/SmolLM2-135M-MemoryFusion-r64


## Reload and verify both final architectures


In [7]:
from tinycenn_lm.smollm2_memory_fusion import (
    MemoryFusionLlamaAttention,
    build_smollm2_memory_fusion,
    structural_summary,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = (
    torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == 'cuda' else torch.float32)
)

for rank in MEMORY_RANKS:
    model = build_smollm2_memory_fusion(OUTPUT_ROOT / f'r{rank}', device=device, dtype=dtype)
    summary = structural_summary(model)
    print(f'r{rank}:', summary)
    assert summary['memory_fusion_layers'] == 30
    assert summary['transformer_attention_layers'] == 0
    assert all(
        layer.self_attn.core.memory_rank == rank
        for layer in model.model.layers
    )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('STRUCTURE: PASS for r48 and r64')


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

r48: {'memory_fusion_layers': 30, 'transformer_attention_layers': 0}


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

r64: {'memory_fusion_layers': 30, 'transformer_attention_layers': 0}
STRUCTURE: PASS for r48 and r64


## Held-out model-level perplexity test

This is deliberately different from the training stream: the comparison uses the same fixed blocks from **WikiText-2 test** for original SmolLM2, r48 and r64. Lower NLL/perplexity is better.


In [8]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

wiki = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
stream = '\n'.join(str(x) for x in wiki['text'] if str(x).strip())
ids = tokenizer(stream, add_special_tokens=False)['input_ids']

EVAL_CONTEXTS = [128, 256]
EVAL_BLOCKS = 12

def make_eval_blocks(context):
    need = context + 1
    blocks = []
    for start in range(0, len(ids) - need, need):
        blocks.append(torch.tensor(ids[start:start+need], dtype=torch.long))
        if len(blocks) >= EVAL_BLOCKS:
            break
    return blocks

@torch.no_grad()
def evaluate_model(model, blocks, context):
    model.eval()
    losses = []
    for block in blocks:
        x = block[:context].unsqueeze(0).to(device)
        out = model(input_ids=x, labels=x, use_cache=False, return_dict=True)
        losses.append(float(out.loss.detach().float()))
    nll = sum(losses) / len(losses)
    return {'nll': nll, 'ppl': math.exp(nll), 'blocks': len(losses)}

eval_rows = []
for context in EVAL_CONTEXTS:
    blocks = make_eval_blocks(context)

    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype).to(device)
    base.config.use_cache = False
    base_result = evaluate_model(base, blocks, context)
    eval_rows.append({'model': 'Transformer original', 'rank': 0, 'context': context, **base_result})
    del base
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    for rank in MEMORY_RANKS:
        student = build_smollm2_memory_fusion(OUTPUT_ROOT / f'r{rank}', device=device, dtype=dtype)
        result = evaluate_model(student, blocks, context)
        eval_rows.append({'model': f'Memory Fusion r{rank}', 'rank': rank, 'context': context, **result})
        del student
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

eval_df = pd.DataFrame(eval_rows)
for context in EVAL_CONTEXTS:
    base_nll = float(eval_df[(eval_df.context == context) & (eval_df['rank'] == 0)].iloc[0].nll)
    mask = eval_df.context == context
    eval_df.loc[mask, 'delta_nll_vs_transformer'] = eval_df.loc[mask, 'nll'] - base_nll
display(eval_df.sort_values(['context', 'nll']))


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/wikitext@b08601e04326c79dfdd32d625aee71d232d685c3/.huggingface.yaml'. Repository id must be 'namespace/name', got 'wikitext'.

## Generation smoke test


In [ ]:
PROMPTS = [
    'The capital of Austria is',
    'Artificial intelligence can help society by',
    'Once upon a time, a small robot was lost in a park.',
]

for rank in MEMORY_RANKS:
    print('\n' + '#' * 100)
    print('MEMORY FUSION r' + str(rank))
    model = build_smollm2_memory_fusion(OUTPUT_ROOT / f'r{rank}', device=device, dtype=dtype)
    model.eval()
    for prompt in PROMPTS:
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=40,
                min_new_tokens=10,
                do_sample=True,
                temperature=0.75,
                top_p=0.90,
                top_k=40,
                repetition_penalty=1.08,
                no_repeat_ngram_size=4,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        print('\n' + '=' * 90)
        print(tokenizer.decode(output[0], skip_special_tokens=True))
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## Final winner summary


In [ ]:
student_rows = eval_df[eval_df['rank'] > 0].copy()
summary = (
    student_rows.groupby(['rank'], as_index=False)
    .agg(mean_delta_nll=('delta_nll_vs_transformer', 'mean'),
         mean_nll=('nll', 'mean'),
         mean_ppl=('ppl', 'mean'))
    .sort_values('mean_delta_nll')
)
display(summary)
winner = int(summary.iloc[0]['rank'])
print('MODEL-LEVEL WINNER:', f'Memory Fusion r{winner}')
for context in EVAL_CONTEXTS:
    row = eval_df[(eval_df['rank'] == winner) & (eval_df.context == context)].iloc[0]
    print(
        f"context={context}: NLL={row.nll:.6f} PPL={row.ppl:.4f} "
        f"ΔNLL vs Transformer={row.delta_nll_vs_transformer:+.6f}"
    )

result_path = OUTPUT_ROOT / 'r48_r64_model_comparison.json'
result_path.write_text(json.dumps({
    'training': reports,
    'evaluation': eval_df.to_dict(orient='records'),
    'winner_rank': winner,
}, indent=2))
print('Saved comparison:', result_path)
